In [3]:
!conda run -n geodiff python -c "import torch; print(torch.__version__, torch.cuda.is_available())"


2.5.1+cu121 True



In [4]:
%cd ~/AttUnode/dataset/GeoDiff

/root/AttUnode/dataset/GeoDiff


In [6]:
script = '''
from rdkit import Chem
from rdkit.Chem import AllChem
import pickle
import sys
sys.path.insert(0, '.')
from utils.datasets import rdmol_to_data

caffeine = Chem.MolFromSmiles('CN1C=NC2=C1C(=O)N(C(=O)N2C)C')
caffeine = Chem.AddHs(caffeine)
AllChem.EmbedMolecule(caffeine, randomSeed=42)

data = rdmol_to_data(caffeine, smiles='CN1C=NC2=C1C(=O)N(C(=O)N2C)C')
with open('caffeine_test.pkl', 'wb') as f:
    pickle.dump([data], f)
print('Built caffeine_test.pkl')
'''

with open("build_caffeine_data.py", "w") as f:
    f.write(script)

In [7]:
!conda run -n geodiff python build_caffeine_data.py

Built caffeine_test.pkl



In [9]:
!conda run -n geodiff python test.py logs/qm9_default/checkpoints/qm9_default.pt \
    --num_confs 10 \
    --test_set caffeine_test.pkl \
    --start_idx 0 --end_idx 1 \
    --w_global 0.3

[Packed] 1 Molecules, 1 Conformations.

/root/AttUnode/dataset/GeoDiff/test.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(args.ckpt)
[2026-09-17 05:

In [10]:
import glob
latest_run = sorted(glob.glob("logs/qm9_default/sample_*"))[-1]
print("Using output folder:", latest_run)

Using output folder: logs/qm9_default/sample_2026_09_17__05_34_14


In [11]:
sdf_script = '''
import pickle
from rdkit import Chem
from rdkit.Geometry import Point3D

with open("{latest_run}/samples_all.pkl", "rb") as f:
    data_list = pickle.load(f)

data = data_list[0]
mol = data.rdmol
num_atoms = mol.GetNumAtoms()
pos_gen = data.pos_gen.numpy().reshape(-1, num_atoms, 3)
print(f"{{pos_gen.shape[0]}} conformers of {{num_atoms}} atoms each")

writer = Chem.SDWriter("caffeine_geodiff_conformers.sdf")
for i in range(pos_gen.shape[0]):
    m = Chem.Mol(mol)
    conf = m.GetConformer() if m.GetNumConformers() > 0 else Chem.Conformer(num_atoms)
    for atom_idx in range(num_atoms):
        x, y, z = pos_gen[i, atom_idx]
        conf.SetAtomPosition(atom_idx, Point3D(float(x), float(y), float(z)))
    if m.GetNumConformers() == 0:
        m.AddConformer(conf, assignId=True)
    m.SetProp("_Name", f"caffeine_GeoDiff_conf_{{i}}")
    writer.write(m)
writer.close()
print("Saved caffeine_geodiff_conformers.sdf")
'''.format(latest_run=latest_run)

with open("save_geodiff_sdf.py", "w") as f:
    f.write(sdf_script)

In [ ]:
!conda run -n geodiff python save_geodiff_sdf.py